# 3. Costed fire stations, written with sets and dictionaries

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 6–8. The third version of the
model: the data are named (stations `'A'` … `'D'`) and kept in dictionaries, and Pyomo builds one constraint per
city from a rule.

**The problem.** Four fire stations $a, b, c, d$ can be run; each covers some of the cities 0–4 and has a
running cost (slides, PDF pages 6–8):

| Station | Covers cities | Cost |
| --- | --- | --- |
| $a$ | 0, 2 | 12 |
| $b$ | 1, 2, 3 | 7 |
| $c$ | 1, 4 | 10 |
| $d$ | 4 | 5 |

Choose stations so that every city is covered by at least one of them, at the lowest total cost. $x_t = 1$ if
station $t$ is run:

minimise $\sum_{t \in S} c_t x_t$ subject to $\sum_{t:\, e \in s_t} x_t \ge 1$ for every city $e$, $x_t \in \{0, 1\}$.

It is the minimum set cover of Week 05 (demo 3), with a cost on each set.

**Numbering.** In this notebook the cities are numbered **1–5**: city 1 here is city 0 on the slides, and so on.
The stations are `'A'` … `'D'` (upper case).

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Cost **24**: stations A, B and D, the same as demos 1 and 2.

*Original script: `5 Source Files/Python/costed_firestation_sets_dictionaries.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model and name the two sets: the stations and the cities.

In [ ]:
# Create the model and the two sets (cities 1-5 here = cities 0-4 on the slides)
model = ConcreteModel()

firestations = ['A','B','C','D']
cities = [1,2,3,4,5]

One binary variable per station, indexed by its name, and the data as dictionaries: the cost of each station and the cities it covers ($s_t$ on the slides).

In [ ]:
# One binary variable per station: x['A'] = 1 if station A is selected
model.x = Var(firestations, within=Binary)

cost = {'A': 12, 'B': 7, 'C': 10, 'D': 5}   # running cost of each station
covers = {          # cities each station covers (s_t on the slides)
    'A': [1,3],
    'B': [2,3,4],
    'C': [2,5],
    'D': [5]
    }

The objective and the constraints. `cover_constraint` is a *rule*: Pyomo calls it once for each city and adds the stations whose `covers` list contains that city.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(cost[station] * model.x[station] for station in firestations), sense=minimize)

# One constraint per city: the stations that cover it, at least one selected
def cover_constraint(model, city):
    return sum(model.x[s] for s in firestations if city in covers[s]) >= 1

model.cover_constraint = Constraint(cities, rule=cover_constraint)

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: one constraint per city, 1–5.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the chosen stations (value 1) and the total cost.

In [ ]:
# Display the results
model.display()

## Try it

A fifth station E could be built: it covers cities 4 and 5 (numbered as in this notebook) and costs 6. Add it
to the data and run all cells again. Is it worth building?

<details><summary>Answer</summary>

Add `'E'` to `firestations`, `'E': 6` to `cost` and `'E': [4, 5]` to `covers`; the objective and the rule do
not change, which is the point of this version. The answer stays A, B, D at cost 24. Only A is forced now
(city 1); for cities 2–5, B + D costs 12, B + E costs 13 and C + E costs 16. E would be worth building only if
it cost less than 5.
</details>